# RAL-CLIP Global Embedding Sweep

Notebook này test bản **RAL-CLIP global-only proxy** khi chưa có local patch tokens.

Theo proposal, RAL-CLIP đầy đủ cần global CLIP embedding + local patch tokens. Hiện chỉ có global embeddings `.pt`, nên notebook này chỉ cài phần làm được:

- source real-only memory bank từ FF++ train REAL
- semantic nearest-real retrieval bằng cosine similarity
- anomaly score bằng khoảng cách tới real-face manifold trong global feature space
- optional safe target real-bank expansion bằng low-score target samples
- unsupervised threshold calibration bằng 2-component GMM

Kết quả dùng để sanity-check hướng real-anchored trước khi chạy RAL-CLIP full với local tokens.

## Kaggle Setup

In [ ]:
# Chạy cell này trên Kaggle nếu repo chưa có trong /kaggle/working.
!git clone -b dev https://github.com/hoavien0110/training-free-tta-for-deepfake-detection.git /kaggle/working/training-free-tta-for-deepfake-detection
%cd /kaggle/working/training-free-tta-for-deepfake-detection
# !pip install -q -e . --no-deps

## Check Inputs

In [ ]:
!ls -lah /kaggle/input
!find /kaggle/input -maxdepth 3 -type f \( -name "*.pt" -o -name "*.csv" \) | sort | sed -n "1,180p"

## Imports

In [ ]:
from pathlib import Path
import json
import sys
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, average_precision_score, confusion_matrix, f1_score, roc_auc_score, roc_curve
from sklearn.mixture import GaussianMixture
from tqdm.auto import tqdm

repo_root = Path.cwd()
if (repo_root / 'code').exists():
    sys.path.insert(0, str(repo_root / 'code'))
else:
    sys.path.insert(0, str(repo_root))

from deepfake_tta.modeling import load_feature_file, seed_everything
from testing.evaluate_tta_matrix import (
    apply_aligned_ids,
    build_aligned_balanced_ids,
    find_feature_files,
    infer_feature_meta,
    select_balanced_subset,
)


## Config

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
BLOCK_SIZE = 16

# Same input layout as 06_eval_tta_best4_ensemble_sweep.ipynb.
FFPP_SPLIT = Path('/kaggle/input/ffpp-split-features')
FFPP_CORR = Path('/kaggle/input/ffpp-test-embeddings/ffpp_test_embeddings')
CELEB_CORR = Path('/kaggle/input/deepfakebench-features')

TRAIN_FEATURES = FFPP_SPLIT / 'ffpp_train_features.pt'
DATASETS = {
    'ffpp-test': FFPP_SPLIT / 'ffpp_test_features.pt',
    'ffpp-test-corruption': FFPP_CORR,
    'celebdfv1-test-corruption': CELEB_CORR,
    'ffpp-test-balanced': FFPP_SPLIT / 'ffpp_test_features.pt',
    'ffpp-test-corruption-balanced': FFPP_CORR,
    'celebdfv1-test-corruption-balanced': CELEB_CORR,
}

BALANCED_DATASETS = {'ffpp-test-balanced'}
BALANCED_ALIGNED_DATASETS = {'ffpp-test-corruption-balanced', 'celebdfv1-test-corruption-balanced'}

# Sweep nhỏ cho RAL-CLIP global-only proxy.
# top_m: số source-real nearest neighbors dùng để tính real-manifold distance.
# score_mode='min': 1 - max cosine; 'mean': mean distance top-M; 'weighted': similarity-weighted distance.
# expand_q: add target samples có score thấp nhất vào real bank rồi score lại. 0.0 = tắt expansion.
RAL_PARAM_SETS = [
    {'param_id': 'global_min_m1_noexp', 'top_m': 1, 'score_mode': 'min', 'expand_q': 0.0},
    {'param_id': 'global_mean_m5_noexp', 'top_m': 5, 'score_mode': 'mean', 'expand_q': 0.0},
    {'param_id': 'global_mean_m16_noexp', 'top_m': 16, 'score_mode': 'mean', 'expand_q': 0.0},
    {'param_id': 'global_weighted_m16_noexp', 'top_m': 16, 'score_mode': 'weighted', 'expand_q': 0.0},
    {'param_id': 'global_min_m1_exp10', 'top_m': 1, 'score_mode': 'min', 'expand_q': 0.10},
    {'param_id': 'global_mean_m16_exp10', 'top_m': 16, 'score_mode': 'mean', 'expand_q': 0.10},
    {'param_id': 'global_weighted_m32_exp10', 'top_m': 32, 'score_mode': 'weighted', 'expand_q': 0.10},
]

MAX_SOURCE_REAL = None       # None = full FF++ real train memory. Set 4096/8192 nếu Kaggle RAM yếu.
MAX_TARGET_REAL_ADD = 2048   # cap target real-bank expansion.
SIM_BATCH_SIZE = 2048        # reduce nếu RAM/GPU yếu.

RESULTS_OUTPUT = Path('/kaggle/working/ral_clip_global_embedding_sweep_results.csv')
PROBES_OUTPUT = Path('/kaggle/working/ral_clip_global_embedding_probes.csv')

seed_everything(SEED)
print('device:', DEVICE)
print('param sets:', len(RAL_PARAM_SETS))

## RAL-CLIP Global Proxy Helpers

In [ ]:
def normalize_features(feats):
    return F.normalize(feats.float(), dim=-1).cpu()

def select_source_real_memory(train_feats, train_labels, max_source_real=None, seed=42):
    labels = train_labels.long()
    real_idx = torch.where(labels.eq(0))[0]
    if max_source_real is not None and len(real_idx) > max_source_real:
        generator = torch.Generator()
        generator.manual_seed(seed)
        real_idx = real_idx[torch.randperm(len(real_idx), generator=generator)[:max_source_real]]
    memory = normalize_features(train_feats[real_idx])
    print('source real memory:', tuple(memory.shape))
    return memory

@torch.inference_mode()
def ral_global_scores(test_feats, memory_feats, *, top_m=16, score_mode='mean', sim_batch_size=2048, device='cpu'):
    test_feats = normalize_features(test_feats)
    memory_feats = normalize_features(memory_feats)
    top_m = min(int(top_m), memory_feats.shape[0])
    memory_dev = memory_feats.to(device)
    scores = []
    for start in tqdm(range(0, len(test_feats), sim_batch_size), leave=False):
        batch = test_feats[start:start + sim_batch_size].to(device)
        sims = batch @ memory_dev.T
        vals = sims.topk(k=top_m, dim=1).values
        dists = 1.0 - vals
        if score_mode == 'min':
            batch_scores = dists[:, 0]
        elif score_mode == 'mean':
            batch_scores = dists.mean(dim=1)
        elif score_mode == 'weighted':
            weights = torch.softmax(vals, dim=1)
            batch_scores = (weights * dists).sum(dim=1)
        else:
            raise ValueError(f'Unknown score_mode={score_mode}')
        scores.append(batch_scores.cpu())
    return torch.cat(scores).numpy()

def expand_memory_with_low_score_targets(memory_feats, test_feats, scores, expand_q, max_add=2048):
    if expand_q <= 0:
        return memory_feats, []
    threshold = float(np.quantile(scores, expand_q))
    selected = np.where(scores <= threshold)[0]
    if max_add is not None:
        selected = selected[:max_add]
    if len(selected) == 0:
        return memory_feats, []
    expanded = torch.cat([memory_feats.cpu(), normalize_features(test_feats[selected])], dim=0).contiguous()
    return expanded, selected.tolist()

def gmm_threshold(scores):
    scores = np.asarray(scores, dtype=np.float64)
    if len(np.unique(scores)) < 3:
        return float(np.median(scores)), None
    gmm = GaussianMixture(n_components=2, random_state=SEED)
    gmm.fit(scores.reshape(-1, 1))
    means = gmm.means_.ravel()
    order = np.argsort(means)
    lo, hi = means[order[0]], means[order[1]]
    grid = np.linspace(scores.min(), scores.max(), 4096)
    logprob = gmm._estimate_weighted_log_prob(grid.reshape(-1, 1))
    diff = logprob[:, order[0]] - logprob[:, order[1]]
    between = (grid >= lo) & (grid <= hi)
    if between.any():
        idxs = np.where(between)[0]
        threshold = grid[idxs[np.argmin(np.abs(diff[idxs]))]]
    else:
        threshold = (lo + hi) / 2
    return float(threshold), gmm

def calculate_eer(y_true, y_score):
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    fnr = 1 - tpr
    idx = np.nanargmin(np.abs(fnr - fpr))
    return float((fpr[idx] + fnr[idx]) / 2), float(thresholds[idx])

def evaluate_anomaly_scores(labels, scores, threshold):
    y_true = labels.detach().cpu().numpy().astype(int)
    y_score = np.asarray(scores, dtype=np.float64)
    y_pred = (y_score > threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    eer, eer_threshold = calculate_eer(y_true, y_score)
    return {
        'acc': accuracy_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'auc': roc_auc_score(y_true, y_score),
        'ap': average_precision_score(y_true, y_score),
        'eer': eer,
        'eer_threshold': eer_threshold,
        'threshold': float(threshold),
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp),
    }

def sample_ids_from_payload(payload, n):
    paths = payload.get('paths')
    if paths is None:
        return [f'idx_{idx:06d}' for idx in range(n)]
    return list(paths)


## Load Source Real Memory

In [ ]:
train_feats, train_labels, train_payload = load_feature_file(str(TRAIN_FEATURES))
source_real_memory = select_source_real_memory(train_feats, train_labels, max_source_real=MAX_SOURCE_REAL, seed=SEED)
print('train counts [REAL, FAKE]:', torch.bincount(train_labels.long(), minlength=2).tolist())

## Run RAL-CLIP Global Sweep

In [ ]:
rows = []
probe_rows = []

for dataset_name, dataset_path in DATASETS.items():
    feature_paths = find_feature_files(Path(dataset_path))
    print('\nDataset', dataset_name)
    for path in feature_paths:
        print(' -', path)

    aligned_ids = None
    if dataset_name in BALANCED_ALIGNED_DATASETS:
        aligned_ids = build_aligned_balanced_ids(feature_paths, BLOCK_SIZE)

    for feature_path in feature_paths:
        feats, labels, payload = load_feature_file(str(feature_path))
        modified_order = False
        if aligned_ids is not None:
            feats, labels = apply_aligned_ids(feats, labels, payload, aligned_ids)
            modified_order = True
        elif dataset_name in BALANCED_DATASETS:
            feats, labels = select_balanced_subset(
                feats,
                labels,
                seed=SEED + sum(ord(ch) for ch in dataset_name + feature_path.name),
                name=f'{dataset_name}/{feature_path.name}',
            )
            modified_order = True

        meta = infer_feature_meta(dataset_name, feature_path, payload)
        sample_ids = [f'idx_{idx:06d}' for idx in range(len(labels))] if modified_order else sample_ids_from_payload(payload, len(labels))

        for cfg in RAL_PARAM_SETS:
            param_id = cfg['param_id']
            top_m = cfg['top_m']
            score_mode = cfg['score_mode']
            expand_q = cfg['expand_q']
            print('\nRAL-CLIP global:', dataset_name, feature_path.name, param_id)

            memory = source_real_memory
            scores = ral_global_scores(
                feats,
                memory,
                top_m=top_m,
                score_mode=score_mode,
                sim_batch_size=SIM_BATCH_SIZE,
                device=DEVICE,
            )
            selected_target_real = []
            if expand_q > 0:
                memory, selected_target_real = expand_memory_with_low_score_targets(
                    memory,
                    feats,
                    scores,
                    expand_q,
                    max_add=MAX_TARGET_REAL_ADD,
                )
                scores = ral_global_scores(
                    feats,
                    memory,
                    top_m=top_m,
                    score_mode=score_mode,
                    sim_batch_size=SIM_BATCH_SIZE,
                    device=DEVICE,
                )

            threshold, _ = gmm_threshold(scores)
            metrics = evaluate_anomaly_scores(labels, scores, threshold)
            rows.append({
                **meta,
                'method': 'ral_clip_global',
                'param_id': param_id,
                'top_m': top_m,
                'score_mode': score_mode,
                'expand_q': expand_q,
                'source_real_memory': int(source_real_memory.shape[0]),
                'expanded_memory': int(memory.shape[0]),
                'target_real_added': int(len(selected_target_real)),
                **metrics,
            })

            preds = (scores > threshold).astype(int)
            for idx, (label, score, pred) in enumerate(zip(labels.detach().cpu().numpy(), scores, preds)):
                probe_rows.append({
                    **meta,
                    'method': 'ral_clip_global',
                    'param_id': param_id,
                    'sample_index': idx,
                    'sample_id': sample_ids[idx] if idx < len(sample_ids) else f'idx_{idx:06d}',
                    'label': int(label),
                    'score': float(score),
                    'pred': int(pred),
                    'threshold': float(threshold),
                })

results = pd.DataFrame(rows)
probes = pd.DataFrame(probe_rows)
RESULTS_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
results.to_csv(RESULTS_OUTPUT, index=False)
probes.to_csv(PROBES_OUTPUT, index=False)
print('saved results:', RESULTS_OUTPUT)
print('saved probes:', PROBES_OUTPUT)
display(results.head())

## Preview Results

In [ ]:
results = pd.read_csv(RESULTS_OUTPUT)
display(results.sort_values(['dataset', 'method', 'f1'], ascending=[True, True, False]).head(60))

metric_cols = ['acc', 'f1', 'auc', 'ap', 'eer']
summary = (
    results.groupby(['dataset', 'method', 'param_id'], dropna=False)[metric_cols]
    .mean()
    .reset_index()
    .sort_values(['dataset', 'f1'], ascending=[True, False])
)
summary_path = Path('/kaggle/working/ral_clip_global_embedding_sweep_summary.csv')
summary.to_csv(summary_path, index=False)
print('saved summary:', summary_path)
display(summary)

## Best Params

In [ ]:
best = (
    summary.sort_values(['dataset', 'f1'], ascending=[True, False])
    .groupby('dataset', as_index=False)
    .head(5)
)
best_path = Path('/kaggle/working/ral_clip_global_embedding_sweep_best.csv')
best.to_csv(best_path, index=False)
print('saved best:', best_path)
display(best)